# Ejercicio 10: Re-ranking

**Objetivo:** Implementar y evaluar un pipeline de Recuperación de Información en dos etapas, y analizar el impacto del re-ranking en la calidad del ranking.

## Parte 1. Preparación del corpus

* Cargar el corpus (documentos/pasajes).
* Cargar las consultas (queries).
* Cargar qrels (relevancia).

**Nombre:** José Armando Sarango Cuenca

In [4]:
%pip install beir


   ---------------------------------------- 0.0/555.1 kB ? eta -:--:--
   ---------------------------------------- 555.1/555.1 kB 5.6 MB/s  0:00:00

   ------ --------------------------------- 1/6 [dill]
   ------ --------------------------------- 1/6 [dill]
   ------ --------------------------------- 1/6 [dill]
   ------------- -------------------------- 2/6 [pytrec-eval-terrier]
   -------------------- ------------------- 3/6 [multiprocess]
   -------------------- ------------------- 3/6 [multiprocess]
   -------------------- ------------------- 3/6 [multiprocess]
   -------------------------- ------------- 4/6 [datasets]
   -------------------------- ------------- 4/6 [datasets]
   -------------------------- ------------- 4/6 [datasets]
   -------------------------- ------------- 4/6 [datasets]
   -------------------------- ------------- 4/6 [datasets]
   -------------------------- ------------- 4/6 [datasets]
   -------------------------- ------------- 4/6 [datasets]
   -----------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from beir import util
from beir.datasets.data_loader import GenericDataLoader
import pandas as pd
import numpy as np

In [6]:
DATASET_NAME = "scifact"
DATA_DIR = "../data/beir_datasets"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{DATASET_NAME}.zip"
util.download_and_unzip(url, DATA_DIR)

../data/beir_datasets\scifact.zip: 100%|██████████| 2.69M/2.69M [00:28<00:00, 98.6kiB/s]


'../data/beir_datasets\\scifact'

In [7]:
dataset_path = DATA_DIR + "/" + DATASET_NAME
corpus, queries, qrels = GenericDataLoader(dataset_path).load(split="test")

100%|██████████| 5183/5183 [00:00<00:00, 85648.63it/s]


In [8]:
df_corpus = (
    pd.DataFrame.from_dict(corpus, orient="index")
      .reset_index()
      .rename(columns={"index": "doc_id"})
)

df_corpus

,doc_id,text,title
0,4983,Alterations of the architecture of cerebral wh...,Microstructural development of human newborn c...
1,5836,Myelodysplastic syndromes (MDS) are age-depend...,Induction of myelodysplasia by myeloid-derived...
2,7912,ID elements are short interspersed elements (S...,"BC1 RNA, the transcript from a master gene for..."
3,18670,DNA methylation plays an important role in bio...,The DNA Methylome of Human Peripheral Blood Mo...
4,19238,Two human Golli (for gene expressed in the oli...,The human myelin basic protein gene is include...
...,...,...,...
5178,195689316,BACKGROUND The main associations of body-mass ...,Body-mass index and cause-specific mortality i...
5179,195689757,A key aberrant biological difference between t...,Targeting metabolic remodeling in glioblastoma...
5180,196664003,A signaling pathway transmits information from...,Signaling architectures that transmit unidirec...
5181,198133135,AIMS Trabecular bone score (TBS) is a surrogat...,"Association between pre-diabetes, type 2 diabe..."


In [9]:
df_queries = (
    pd.DataFrame.from_dict(queries, orient="index", columns=["query"])
      .reset_index()
      .rename(columns={"index": "query_id"})
)

df_queries

,query_id,query
0,1,0-dimensional biomaterials show inductive prop...
1,3,"1,000 genomes project enables mapping of genet..."
2,5,1/2000 in UK have abnormal PrP positivity.
3,13,5% of perinatal mortality is due to low birth ...
4,36,A deficiency of vitamin B12 increases blood le...
...,...,...
295,1379,Women with a higher birth weight are more like...
296,1382,aPKCz causes tumour enhancement by affecting g...
297,1385,cSMAC formation enhances weak ligand signalling.
298,1389,mTORC2 regulates intracellular cysteine levels...


In [10]:
rows = []
for qid, docs in qrels.items():
    for doc_id, rel in docs.items():
        rows.append({
            "query_id": qid,
            "doc_id": doc_id,
            "relevance": rel
        })

df_qrels = pd.DataFrame(rows)
df_qrels

,query_id,doc_id,relevance
0,1,31715818,1
1,3,14717500,1
2,5,13734012,1
3,13,1606628,1
4,36,5152028,1
...,...,...,...
334,1379,17450673,1
335,1382,17755060,1
336,1385,306006,1
337,1389,23895668,1


In [11]:
# Elegimos una query cualquiera que tenga varios documentos relevantes
qid = "133"

print("Query:")
print(df_queries.loc[df_queries["query_id"] == qid, "query"].values[0])

print("\nDocumentos relevantes para esta query:")
df_qrels[(df_qrels["query_id"] == qid) & (df_qrels["relevance"] > 0)]

Query:
Assembly of invadopodia is triggered by focal generation of phosphatidylinositol-3,4-biphosphate and the activation of the nonreceptor tyrosine kinase Src.

Documentos relevantes para esta query:


,query_id,doc_id,relevance
31,133,38485364,1
32,133,6969753,1
33,133,17934082,1
34,133,16280642,1
35,133,12640810,1


## Parte 2. Retrieval inicial (baseline)

* Implementar retrieval inicial con BM25
* Obtener métricas: Recall@10 nDCG@10

In [26]:
from rank_bm25 import BM25Okapi
from beir.retrieval.evaluation import EvaluateRetrieval
doc_ids = list(corpus.keys())
tokenized_corpus = [
    (corpus[doc_id]["title"] + " " + corpus[doc_id]["text"]).lower().split()
    for doc_id in doc_ids
]

bm25 = BM25Okapi(tokenized_corpus)

#2. Recuperar top-10 por query
results_bm25 = {}

for qid, query_text in queries.items():
    tokenized_query = query_text.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_k_indices = np.argsort(scores)[::-1][:10]
    results_bm25[qid] = {
        doc_ids[i]: float(scores[i]) for i in top_k_indices
    }

# 3. Métricas baseline
evaluator = EvaluateRetrieval()
ndcg, _map, recall, precision = evaluator.evaluate(qrels, results_bm25, k_values=[10])

print("BM25 Baseline")
print(f"Recall@10 : {recall['Recall@10']:.4f}")
print(f"nDCG@10   : {ndcg['NDCG@10']:.4f}")

BM25 Baseline
Recall@10 : 0.6862
nDCG@10   : 0.5597


## Parte 3. Implementación del re-ranking _cross-encoder_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [27]:
from sentence_transformers import CrossEncoder

#1. Cargar cross-encoder preentrenado
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

#2. Re-rankear top-10 de BM25 con el cross-encoder 
results_ce = {}

for qid, query_text in queries.items():
    candidates = results_bm25[qid] 
    doc_id_list = list(candidates.keys())

    # Construir pares para el cross-encoder
    pairs = [
        (query_text, corpus[doc_id]["title"] + " " + corpus[doc_id]["text"])
        for doc_id in doc_id_list
    ]

    # Puntuar con el cross-encoder (s2: scorer preciso)
    ce_scores = cross_encoder.predict(pairs)

    results_ce[qid] = {
        doc_id_list[i]: float(ce_scores[i]) for i in range(len(doc_id_list))
    }

# 3. Métricas post re-ranking
ndcg_ce, _map_ce, recall_ce, _ = evaluator.evaluate(qrels, results_ce, k_values=[10])

print("Cross-Encoder Re-ranking")
print(f"Recall@10 : {recall_ce['Recall@10']:.4f}")
print(f"nDCG@10   : {ndcg_ce['NDCG@10']:.4f}")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3327.08it/s]


Cross-Encoder Re-ranking
Recall@10 : 0.6862
nDCG@10   : 0.6172


In [28]:
#Identificar documentos que cambian de posición

def get_ranking(results, qid):
    sorted_docs = sorted(results[qid].items(), key=lambda x: x[1], reverse=True)
    return [doc_id for doc_id, _ in sorted_docs]

print("=== Cambios de posición en top-10 (ejemplo query '133') ===\n")
qid_ejemplo = "133"
ranking_bm25 = get_ranking(results_bm25, qid_ejemplo)
ranking_ce   = get_ranking(results_ce,   qid_ejemplo)
rows = []
for doc_id in set(ranking_bm25 + ranking_ce):
    pos_bm25 = ranking_bm25.index(doc_id) + 1 if doc_id in ranking_bm25 else "-"
    pos_ce   = ranking_ce.index(doc_id)   + 1 if doc_id in ranking_ce   else "-"
    delta = "-"
    if isinstance(pos_bm25, int) and isinstance(pos_ce, int):
        delta = pos_bm25 - pos_ce  
    rows.append({"doc_id": doc_id, "BM25": pos_bm25, "CrossEncoder": pos_ce, "Δ (subió+)": delta})

df_cambios = pd.DataFrame(rows).sort_values("CrossEncoder")
print(df_cambios.to_string(index=False))

=== Cambios de posición en top-10 (ejemplo query '133') ===

  doc_id  BM25  CrossEncoder  Δ (subió+)
12640810     6             1           5
 6969753    10             2           8
 9507605     2             3          -1
86694016     8             4           4
17934082     9             5           4
37964706     3             6          -3
12785130     5             7          -2
26688294     1             8          -7
30861948     7             9          -2
 5270265     4            10          -6


## Parte 4. Implementación del re-ranking _LTR_

* Re-rankear los top-k candidatos para cada query.
* Identificar qué documentos cambian de posición en el top 10

In [19]:
%pip install lightgbm


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------------------------------ --- 1.3/1.5 MB 21.9 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 15.1 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [29]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split

#Construir features para cada par (query, doc)
def build_features(qid, doc_id, query_text, bm25_score):
    doc = corpus[doc_id]
    doc_text  = (doc["title"] + " " + doc["text"]).lower()
    doc_words = doc_text.split()
    q_words   = set(query_text.lower().split())

    return {
        "bm25_score"     : bm25_score,
        "doc_length"     : len(doc_words),
        "title_length"   : len(doc["title"].split()),
        "query_term_overlap" : len(q_words & set(doc_words)) / (len(q_words) + 1e-9),
        "title_overlap"  : len(q_words & set(doc["title"].lower().split())) / (len(q_words) + 1e-9),
    }

rows = []
for qid, doc_scores in results_bm25.items():
    query_text = queries[qid]
    for doc_id, bm25_score in doc_scores.items():
        feats = build_features(qid, doc_id, query_text, bm25_score)
        # Label de relevancia desde qrels (0 si no aparece)
        label = qrels.get(qid, {}).get(doc_id, 0)
        rows.append({"qid": qid, "doc_id": doc_id, "label": label, **feats})

df_ltr = pd.DataFrame(rows)
print(f"Dataset LTR: {df_ltr.shape[0]} pares (query, doc)")
print(df_ltr.head())

Dataset LTR: 3000 pares (query, doc)
  qid    doc_id  label  bm25_score  doc_length  title_length  \
0   1    825728      0    9.489886         133            11   
1   1  10931595      0    8.945226         221             5   
2   1  43385013      0    7.586050         269            15   
3   1  13231899      0    7.439832         260            14   
4   1  18953920      0    7.035655         158            10   

   query_term_overlap  title_overlap  
0                 0.4            0.0  
1                 0.4            0.0  
2                 0.4            0.2  
3                 0.2            0.0  
4                 0.4            0.0  


In [30]:
# 2. Preparar datos para LightGBM LambdaRank
feature_cols = ["bm25_score", "doc_length", "title_length",
                "query_term_overlap", "title_overlap"]

# Split por query (no por fila) para evitar data leakage
qids_unique = df_ltr["qid"].unique()
qids_train, qids_test = train_test_split(qids_unique, test_size=0.3, random_state=42)

df_train = df_ltr[df_ltr["qid"].isin(qids_train)].copy()
df_test  = df_ltr[df_ltr["qid"].isin(qids_test)].copy()

X_train = df_train[feature_cols].values
y_train = df_train["label"].values
g_train = df_train.groupby("qid").size().values  # group sizes para LTR

X_test  = df_test[feature_cols].values
y_test  = df_test["label"].values
g_test  = df_test.groupby("qid").size().values

#3. Entrenar modelo LambdaRank 
train_data = lgb.Dataset(X_train, label=y_train, group=g_train)
test_data  = lgb.Dataset(X_test,  label=y_test,  group=g_test, reference=train_data)

params = {
    "objective"    : "lambdarank",
    "metric"       : "ndcg",
    "ndcg_eval_at" : [10],
    "learning_rate": 0.05,
    "num_leaves"   : 31,
    "min_data_in_leaf": 1,
    "verbose"      : -1,
}

model_ltr = lgb.train(
    params,
    train_data,
    num_boost_round=100,
    valid_sets=[test_data],
    callbacks=[lgb.early_stopping(10), lgb.log_evaluation(20)],
)

print("\nModelo LTR entrenado.")

Training until validation scores don't improve for 10 rounds
[20]	valid_0's ndcg@10: 0.8548
Early stopping, best iteration is:
[16]	valid_0's ndcg@10: 0.863517

Modelo LTR entrenado.


In [31]:
# 4. Re-rankear con LTR
results_ltr = {}

for qid, doc_scores in results_bm25.items():
    query_text = queries[qid]
    doc_id_list = list(doc_scores.keys())

    feats = np.array([
        list(build_features(qid, doc_id, query_text, doc_scores[doc_id]).values())
        for doc_id in doc_id_list
    ])

    ltr_scores = model_ltr.predict(feats)
    results_ltr[qid] = {
        doc_id_list[i]: float(ltr_scores[i]) for i in range(len(doc_id_list))
    }

# 5. Métricas post re-ranking LTR 
ndcg_ltr, _map_ltr, recall_ltr, _ = evaluator.evaluate(qrels, results_ltr, k_values=[10])

print("=== LTR Re-ranking ===")
print(f"Recall@10 : {recall_ltr['Recall@10']:.4f}")
print(f"nDCG@10   : {ndcg_ltr['NDCG@10']:.4f}")

=== LTR Re-ranking ===
Recall@10 : 0.6862
nDCG@10   : 0.6454


In [32]:
print("=== Cambios de posición BM25 → LTR (query '133') ===\n")
ranking_ltr = get_ranking(results_ltr, qid_ejemplo)

rows_ltr = []
for doc_id in set(ranking_bm25 + ranking_ltr):
    pos_bm25 = ranking_bm25.index(doc_id) + 1 if doc_id in ranking_bm25 else "-"
    pos_ltr  = ranking_ltr.index(doc_id)  + 1 if doc_id in ranking_ltr  else "-"
    delta = "-"
    if isinstance(pos_bm25, int) and isinstance(pos_ltr, int):
        delta = pos_bm25 - pos_ltr
    rows_ltr.append({"doc_id": doc_id, "BM25": pos_bm25, "LTR": pos_ltr, "Δ (subió+)": delta})

df_cambios_ltr = pd.DataFrame(rows_ltr).sort_values("LTR")
print(df_cambios_ltr.to_string(index=False))

=== Cambios de posición BM25 → LTR (query '133') ===

  doc_id  BM25  LTR  Δ (subió+)
17934082     9    1           8
26688294     1    2          -1
 5270265     4    3           1
86694016     8    4           4
12785130     5    5           0
30861948     7    6           1
37964706     3    7          -4
 9507605     2    8          -6
12640810     6    9          -3
 6969753    10   10           0


## Parte 5. Evaluación post re-ranking

Calcular métricas:
* nDCG@10
* MAP
* Recall@10

In [33]:
# 1. Calcular las métricas detalladas para Cross-Encoder y LTR usando el evaluador de BEIR
ndcg_ce, map_ce, recall_ce, precision_ce = evaluator.evaluate(qrels, results_ce, k_values=[10])
ndcg_ltr, map_ltr, recall_ltr, precision_ltr = evaluator.evaluate(qrels, results_ltr, k_values=[10])
metricas_globales = {
    "Sistema (Model)": ["BM25 Baseline", "Cross-Encoder Reranker", "LTR (LambdaRank)"],
    "nDCG@10": [ndcg["NDCG@10"], ndcg_ce["NDCG@10"], ndcg_ltr["NDCG@10"]],
    "MAP@10": [_map["MAP@10"], map_ce["MAP@10"], map_ltr["MAP@10"]],
    "Recall@10": [recall["Recall@10"], recall_ce["Recall@10"], recall_ltr["Recall@10"]]
}
df_evaluacion = pd.DataFrame(metricas_globales)

print("=== TABLA COMPARATIVA DE RENDIMIENTO GLOBAL ===")
df_evaluacion.style.format({
    "nDCG@10": "{:.4f}",
    "MAP@10": "{:.4f}",
    "Recall@10": "{:.4f}"
}).hide(axis="index")

=== TABLA COMPARATIVA DE RENDIMIENTO GLOBAL ===


Sistema (Model),nDCG@10,MAP@10,Recall@10
BM25 Baseline,0.5597,0.5147,0.6862
Cross-Encoder Reranker,0.6172,0.5882,0.6862
LTR (LambdaRank),0.6454,0.6263,0.6862
